# Stock Bot Walk-Forward
Runs historical validation only. It does not use Alpaca credentials or publish a live config.

In [ ]:
# Mount Drive so snapshots and checkpoints survive a Colab disconnect.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Change SNAPSHOT_NAME to the file created on your computer.
REPO_URL = 'https://github.com/uwuexdeemeow/Stock-Market-AI-Bot.git'
SNAPSHOT_NAME = 'CHANGE_ME.tar.gz'
DRIVE_DIR = '/content/drive/MyDrive/StockBotWalkforward'
# Which jobs to run.  RUN_WALKFORWARD is the nested walk-forward below.
# RUN_PHASE_LUCK runs the rebalance-day luck diagnostic and the H-tranche
# preview (research_evidence/phase_luck_20260926/); it saves to a separate
# Drive file, so it never overwrites a walk-forward result.
RUN_WALKFORWARD = True
RUN_PHASE_LUCK = False
# RUN_BAKEOFF runs the pre-registered signal bake-off (Hypothesis H-bakeoff,
# research_evidence/signal_bakeoff_20260926/).  The snapshot must contain the
# 11 sector ETF files (XLK ... XLC); see the notebook doc.  Separate Drive file.
RUN_BAKEOFF = False
GRID_FLAG = '--low-turnover-grid'
OUTPUT_PREFIX = 'wf_delay_robust_lowturnover_20260926'
# Rules fixed in Documentation/DELAY_STRESS_PAPER_ADVISORY.md: select on outer
# years up to 2022 only, and also score every inner fold with fills one
# trading day late (the candidate keeps its worse result).
END_YEAR = 2022
SELECTION_ENTRY_DELAY_DAYS = 1


In [ ]:
# Clone the exact code version and verify the uploaded snapshot checksum.
import hashlib, json, os, pathlib, shutil, subprocess, tarfile
snapshot = pathlib.Path(DRIVE_DIR) / SNAPSHOT_NAME
manifest = snapshot.with_suffix('.manifest.json')
info = json.loads(manifest.read_text())
digest = hashlib.sha256(snapshot.read_bytes()).hexdigest()
assert digest == info['sha256'], 'Snapshot checksum does not match'
repo = pathlib.Path('/content/stockbot')
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
subprocess.run(['git', 'checkout', info['git_commit']], cwd=repo, check=True)
with tarfile.open(snapshot, 'r:gz') as handle: handle.extractall(repo)
print('Snapshot verified:', info['file_count'], 'files')


In [ ]:
# Install the same project dependencies used locally.
subprocess.run(['python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=repo, check=True)
# A saved snapshot is always days old by the time Colab runs it, and this
# research only uses history. Keep every structural check but switch off
# the live-trading freshness rule (--strict would reject any old snapshot).
subprocess.run(['python3', 'factor_data_health.py', '--ready-only', '--no-write', '--warn-days', '100000', '--block-days', '100000'], cwd=repo, check=True)


In [ ]:
# Run two outer folds per process. Copy checkpoints to Drive after each batch.
import multiprocessing
checkpoint = repo / 'signals/walkforward_checkpoint_core_alpha.json'
drive_checkpoint = pathlib.Path(DRIVE_DIR) / checkpoint.name
final_json = repo / f'signals/{OUTPUT_PREFIX}.json'
if RUN_WALKFORWARD:
    if drive_checkpoint.exists(): shutil.copy2(drive_checkpoint, checkpoint)
    workers = max(1, min(8, multiprocessing.cpu_count()))
    for batch in range(10):
        cmd = ['python3', 'core_satellite_nested_walkforward.py', '--strategy', 'core-alpha', GRID_FLAG, '--resume', '--exit-after-folds', '2', '--workers', str(workers), '--output-prefix', OUTPUT_PREFIX, '--no-publish-live-config',
               '--end-year', str(END_YEAR), '--selection-entry-delay-days', str(SELECTION_ENTRY_DELAY_DAYS)]
        subprocess.run(cmd, cwd=repo, check=True)
        if checkpoint.exists(): shutil.copy2(checkpoint, drive_checkpoint)
        if final_json.exists(): break
    assert final_json.exists(), 'Walk-forward did not finish within 10 batches'
    print('Walk-forward complete:', final_json)
else:
    print('RUN_WALKFORWARD is False: walk-forward skipped')


In [ ]:
# Analyze and package the result for review on the project computer.
if RUN_WALKFORWARD:
    csv_path = repo / f'signals/{OUTPUT_PREFIX}.csv'
    subprocess.run(['python3', 'walkforward_analyzer.py', '--csv', str(csv_path), '--json'], cwd=repo, check=True)
    result_archive = pathlib.Path(DRIVE_DIR) / 'stockbot_colab_result.tar.gz'
    with tarfile.open(result_archive, 'w:gz') as handle:
        for path in [final_json, csv_path, csv_path.with_suffix('.analyzer.json'), repo / 'signals/research_run_manifest.json']:
            if path.exists(): handle.add(path, arcname=path.relative_to(repo))
    print('Saved result:', result_archive)


In [ ]:
# Optional: rebalance-day luck diagnostic + H-tranche preview (research only).
# Each script takes about 6 minutes.  Both read only the snapshot data and
# the approved incumbent config; nothing is published or approved.
if RUN_PHASE_LUCK:
    evidence = repo / 'research_evidence/phase_luck_20260926'
    for script in ('phase_luck.py', 'tranche_preview.py'):
        subprocess.run(['python3', str(evidence / script)], cwd=repo, check=True)
    phase_archive = pathlib.Path(DRIVE_DIR) / 'stockbot_phase_luck_result.tar.gz'
    with tarfile.open(phase_archive, 'w:gz') as handle:
        for path in [evidence / 'phase_luck.json', evidence / 'tranche_preview.json']:
            handle.add(path, arcname=path.relative_to(repo))
    print('Saved phase-luck result:', phase_archive)
    print(json.dumps({k: v for k, v in json.loads((evidence / 'tranche_preview.json').read_text()).items() if k != 'books'}, indent=1))
else:
    print('RUN_PHASE_LUCK is False: phase-luck research skipped')


In [ ]:
# Optional: pre-registered signal bake-off (Hypothesis H-bakeoff, research only).
# 240 engine runs, about 20-30 minutes.  Reads only the snapshot data and the
# approved incumbent config; nothing is published or approved.
if RUN_BAKEOFF:
    bakeoff_dir = repo / 'research_evidence/signal_bakeoff_20260926'
    sector_etfs = ['XLK', 'XLY', 'XLF', 'XLV', 'XLE', 'XLI', 'XLP', 'XLU', 'XLRE', 'XLB', 'XLC']
    missing = [t for t in sector_etfs if not (repo / 'data' / f'{t}.parquet').exists()]
    assert not missing, f'Snapshot lacks sector ETF files {missing}: run refresh_etf_data.py for them, then make a new snapshot'
    subprocess.run(['python3', str(bakeoff_dir / 'signal_bakeoff.py')], cwd=repo, check=True)
    bakeoff_json = bakeoff_dir / 'signal_bakeoff.json'
    bakeoff_archive = pathlib.Path(DRIVE_DIR) / 'stockbot_bakeoff_result.tar.gz'
    with tarfile.open(bakeoff_archive, 'w:gz') as handle:
        handle.add(bakeoff_json, arcname=bakeoff_json.relative_to(repo))
    print('Saved bake-off result:', bakeoff_archive)
    result = json.loads(bakeoff_json.read_text())['result']
    print(json.dumps({k: result[k] for k in ('eligible', 'winner', 'reason')}, indent=1))
else:
    print('RUN_BAKEOFF is False: signal bake-off skipped')
